<a href="https://colab.research.google.com/github/Tuantu23-ueh/UEH_ML_Energy_GreenAI_Group_1/blob/main/03_paper1_experiments_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **PAPER 1 - EXPERIMENTS AND GREEN AI**
### **Notebook Structure Overview**
1. **Section 1: Environment Configuration & Reproducibility Protocol**
2. **Section 2: Data Ingestion & Feature Configuration**
3. **Section 3: Green AI Engine**
4. **Section 4: Synthetic Anomaly Injection**
5. **Section 5: Metrics Aggregation & Export**

### **CÔNG VIỆC (SUB-TEAM 2 - HẠN 06/11)**

* **Quốc Huy:**
  1. **Huấn luyện mô hình & Feature Ablation:** Lập trình huấn luyện **Random Forest Regressor** trên 4 bộ đặc trưng (Set A, B, W, C) cho 12 tòa nhà và chạy thực nghiệm Ablation từng đặc trưng độ trễ.
  2. **Tạo dữ liệu lãng phí điện (Paper 2):** Lập trình chèn 3% bất thường nhân tạo (Daytime Spikes & Nocturnal Leaks) và xuất file `synthetic_anomalies_12_clean.csv`.
* **Minh Đức:**
  1. **Đo chi phí Green AI:** Nhúng hàm đo tài nguyên (Thời gian huấn luyện, Latency, Kích thước model MB, RAM) và tinh chỉnh siêu tham số với `TimeSeriesSplit`.
  2. **Đóng gói & Nộp bài:** Tính chỉ số Trung vị & IQR và các chỉ số đánh giá mô hình MAE, RMSE, R2, kiểm thử toàn bộ notebook (`Run All Cells`), tạo Pull Request (PR) trên GitHub và báo cáo cho trưởng nhóm (Tú).

# **SECTION 1: ENVIRONMENT & REPRODUCIBILITY**

> **Note:** Fixes global pseudo-random seeds across Python, NumPy, and Scikit-Learn libraries to guarantee experimental reproducibility.

In [ ]:
# ==============================================================================
# SECTION 1: ENVIRONMENT CONFIGURATION & REPRODUCIBILITY PROTOCOL
# ==============================================================================
import os
import time
import random
import joblib
import psutil
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(SEED)


# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/[UEH_ML]_Energy_GreenAI_Group_1/02_Data/Processed/buildings_12_clean.csv"
DATA_OUT_DIR = "/content/drive/MyDrive/[UEH_ML]_Energy_GreenAI_Group_1/03_Code_Colab/paper1_results"
os.makedirs(DATA_OUT_DIR, exist_ok=True)
print(f"Environment is Ready. Seed fixed: {SEED}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment is Ready. Seed fixed: 42


# **SECTION 2: DATA INGESTION & FEATURE SUBSET SPECIFICATIONS**
> **Note:** Ingests the BDG2 dataset comprising 12 selected buildings (210,528 hourly records). Eliminates lag-initialization `NaN` rows and specifies four feature subsets (Set A, Set B, Set W, Set C) to evaluate feature importance.

In [ ]:
# ==============================================================================
# 2. DATA INGESTION & FEATURE SET CONFIGURATION
# ==============================================================================
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Drop rows with NaN caused by initial lag extraction window (168-hour initialization)
df_clean = df.dropna(subset=['y_kwh', 'lag_1', 'lag_24', 'lag_168', 'roll_mean_24']).copy()

# Chronological Split Check (No shuffling to prevent temporal data leakage)
train_df = df_clean[df_clean['split'] == 'train'].copy()
val_df   = df_clean[df_clean['split'] == 'val'].copy()
test_df  = df_clean[df_clean['split'] == 'test'].copy()

print(f"Dataset Loaded Successfully:")
print(f" - Train Partition (2016)      : {train_df.shape} samples")
print(f" - Validation Partition (H1 2017): {val_df.shape} samples")
print(f" - Test Partition (H2 2017)      : {test_df.shape} samples")

# Define 4 Feature Subsets according to Paper 1 Research Framework
SET_A_TIME = ['hour', 'dayofweek', 'is_weekend', 'month', 'hour_sin', 'hour_cos']
SET_B_TIME_LAG = SET_A_TIME + ['lag_1', 'lag_24', 'lag_168', 'roll_mean_24']
SET_W_TIME_WEATHER = SET_A_TIME + ['air_temperature', 'dew_temperature', 'wind_speed', 'precip_depth_1_hr']
SET_C_FULL = SET_B_TIME_LAG + ['air_temperature', 'dew_temperature', 'wind_speed', 'precip_depth_1_hr']

FEATURE_SETS = {
    'Set_A': SET_A_TIME,
    'Set_B': SET_B_TIME_LAG,
    'Set_W': SET_W_TIME_WEATHER,
    'Set_C': SET_C_FULL
}

Dataset Loaded Successfully:
 - Train Partition (2016)      : (102549, 18) samples
 - Validation Partition (H1 2017): (51341, 18) samples
 - Test Partition (H2 2017)      : (51824, 18) samples


# **SECTION 3: GREEN AI ENGINE**
> **Note:** Huấn luyện Linear Regression, Decision Tree và Random Forest trên 12 tòa nhà × 4 bộ đặc trưng (Set A, B, W, C). Siêu tham số chọn bằng `TimeSeriesSplit` (3 fold, expanding window) trên tập **Train**; tập Test chỉ dùng để báo cáo. Đo chi phí Green AI: thời gian huấn luyện, độ trễ suy luận, kích thước mô hình (MB) và RAM. Mô hình mặc định của Huy (3.1) làm mốc so sánh với bản tinh chỉnh (3.3). Gồm Ablation Random Forest (Huy) và bản mở rộng (Đức).
>
> **Chạy trên Colab:** thư mục nhóm `[UEH_ML]_Energy_GreenAI_Group_1` phải nằm trong "Drive của tôi" (nếu đang ở mục "Được chia sẻ với tôi" thì tạo lối tắt: Sắp xếp → Thêm lối tắt). Thời gian chạy toàn bộ notebook khoảng 40-60 phút; thời gian và dung lượng phụ thuộc máy, nên mọi so sánh chi phí phải chạy cùng một phiên Colab.

In [ ]:
# ==============================================================================
# 3.0 BỔ SUNG THƯ VIỆN, ĐƯỜNG DẪN VÀ HẰNG SỐ CHO GREEN AI ENGINE
# ==============================================================================
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit

ROOT = "/content/drive/MyDrive/[UEH_ML]_Energy_GreenAI_Group_1"
OUT_DIR = DATA_OUT_DIR   # cùng thư mục xuất kết quả với Section 1
LAGS    = ['lag_1', 'lag_24', 'lag_168', 'roll_mean_24']
WEATHER = ['air_temperature', 'dew_temperature', 'wind_speed', 'precip_depth_1_hr']

### **3.1 MODEL PALETTE BENCHMARKING (LR, DT, RF) - mô hình mặc định, không tinh chỉnh (Huy)**
> **Note:** Trains and evaluates Linear Regression, Decision Tree Regressor, and Random Forest Regressor across 12 individual buildings and 4 feature subsets. Evaluates goodness-of-fit ($R^2$, $MAE$, $RMSE$) and ASHRAE Guideline 14 normalized error ($CV(RMSE)$).

In [ ]:
# ==============================================================================
# 3. EVALUATION METRICS & MULTI-MODEL BENCHMARKING RUNNER
# ==============================================================================
def calculate_ashrae_metrics(y_true, y_pred, y_train_mean):
    """Calculates forecast accuracy metrics per ASHRAE Guideline 14 standards."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cv_rmse = (rmse / y_train_mean) * 100.0  # CV(RMSE) % normalized by train mean
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_RMSE_%': cv_rmse}

buildings = df_clean['building_id'].unique()
paper1_results = []

print("Starting Model Palette Benchmarking across 12 buildings...")

for b_id in buildings:
    b_train = train_df[train_df['building_id'] == b_id]
    b_test  = test_df[test_df['building_id'] == b_id]

    y_train = b_train['y_kwh'].values
    y_test  = b_test['y_kwh'].values
    y_train_mean = np.mean(y_train)

    # Iterate over all 4 feature subsets
    for set_name, feature_cols in FEATURE_SETS.items():
        # Scale features strictly on Training Set to avoid look-ahead bias
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(b_train[feature_cols])
        X_test_scaled  = scaler.transform(b_test[feature_cols])

        # 1. Linear Regression
        lr = LinearRegression()
        lr.fit(X_train_scaled, y_train)
        pred_lr = lr.predict(X_test_scaled)
        m_lr = calculate_ashrae_metrics(y_test, pred_lr, y_train_mean)
        paper1_results.append({'Building': b_id, 'Model': 'Linear Regression', 'Set': set_name, **m_lr})

        # 2. Decision Tree Regressor (Unscaled features)
        dt = DecisionTreeRegressor(random_state=SEED)
        dt.fit(b_train[feature_cols], y_train)
        pred_dt = dt.predict(b_test[feature_cols])
        m_dt = calculate_ashrae_metrics(y_test, pred_dt, y_train_mean)
        paper1_results.append({'Building': b_id, 'Model': 'Decision Tree', 'Set': set_name, **m_dt})

        # 3. Random Forest Regressor (Ensemble Baseline)
        rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
        rf.fit(b_train[feature_cols], y_train)
        pred_rf = rf.predict(b_test[feature_cols])
        m_rf = calculate_ashrae_metrics(y_test, pred_rf, y_train_mean)
        paper1_results.append({'Building': b_id, 'Model': 'Random Forest', 'Set': set_name, **m_rf})

results_df = pd.DataFrame(paper1_results)
print(f"Completed benchmarking! Total experiments executed: {len(results_df)}")

Starting Model Palette Benchmarking across 12 buildings...
Completed benchmarking! Total experiments executed: 144


In [ ]:
# ==============================================================================
# SECTION 3.2: METRICS & COST HELPERS
# ==============================================================================
def calculate_metrics(y_true, y_pred, y_train_mean):
    # Giống baseline: CV(RMSE) chia cho trung bình tập train
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    cv_rmse = (rmse / y_train_mean) * 100.0
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_RMSE_%': cv_rmse}

def model_size_mb(obj):
    # Dung lượng mô hình đã serialize (pickle), đơn vị MB
    return len(pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)) / 1e6

def inference_ms_per_sample(predict_fn, n_samples, n_repeat=3):
    # Median của n_repeat lần dự báo trên toàn bộ tập test, quy đổi ms/mẫu
    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        predict_fn()
        times.append(time.perf_counter() - t0)
    return float(np.median(times)) / n_samples * 1000.0

def iqr(x):
    return np.percentile(x, 75) - np.percentile(x, 25)

# Lưới siêu tham số nhỏ; lưới của DT có chứa cấu hình mặc định của baseline (max_depth=None, min_samples_leaf=1)
GRIDS = {
    'Decision Tree': [dict(max_depth=d, min_samples_leaf=l) for d in (5, 10, 20, None) for l in (1, 10, 50)],
    'Random Forest': [dict(max_depth=d, min_samples_leaf=l) for d in (10, None) for l in (1, 10)],
}

def make_model(name, params):
    if name == 'Linear Regression':
        return LinearRegression()
    if name == 'Decision Tree':
        return DecisionTreeRegressor(random_state=SEED, **params)
    if name == 'Random Forest':
        return RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, **params)
    raise ValueError(name)

N_SPLITS = 3   # TimeSeriesSplit: expanding window, luôn huấn luyện trên quá khứ và kiểm tra trên giai đoạn sau đó

def fit_select(name, cols, b_train, grid=None):
    # Chọn siêu tham số bằng TimeSeriesSplit TRÊN TẬP TRAIN (RMSE trung bình các fold), rồi fit lại trên toàn bộ Train.
    # Val và Test không tham gia việc chọn.
    grid = grid if grid is not None else GRIDS.get(name, [{}])
    b_train = b_train.sort_values('timestamp')
    X_raw, y = b_train[cols], b_train['y_kwh'].values
    scaler = StandardScaler().fit(X_raw) if name == 'Linear Regression' else None
    prep = scaler.transform if scaler is not None else (lambda X: X)
    best_params, best_cv = grid[0], None
    if len(grid) > 1:
        tscv = TimeSeriesSplit(n_splits=N_SPLITS)
        for params in grid:
            errs = []
            for tr_idx, va_idx in tscv.split(X_raw):
                m = make_model(name, params)
                m.fit(X_raw.iloc[tr_idx], y[tr_idx])
                errs.append(np.sqrt(mean_squared_error(y[va_idx], m.predict(X_raw.iloc[va_idx]))))
            cv_rmse = float(np.mean(errs))
            if best_cv is None or cv_rmse < best_cv:
                best_cv, best_params = cv_rmse, params
    model = make_model(name, best_params)
    t0 = time.perf_counter()
    model.fit(prep(X_raw), y)
    fit_s = time.perf_counter() - t0
    return dict(model=model, scaler=scaler, prep=prep, params=best_params, fit_s=fit_s, cv_rmse=best_cv)

In [ ]:
# ==============================================================================
# SECTION 3.3: LR / DT / RF x SET A, B, W, C  (12 buildings; siêu tham số chọn bằng TimeSeriesSplit trên Train)
# ==============================================================================
MODEL_NAMES = ['Linear Regression', 'Decision Tree', 'Random Forest']
buildings = df_clean['building_id'].unique()
results_list = []

for b_id in buildings:
    b_train = train_df[train_df['building_id'] == b_id]
    b_test  = test_df[test_df['building_id'] == b_id]
    y_test = b_test['y_kwh'].values
    y_train_mean = b_train['y_kwh'].mean()

    for set_name, cols in FEATURE_SETS.items():
        for name in MODEL_NAMES:
            best = fit_select(name, cols, b_train)
            X_te = best['prep'](b_test[cols])
            y_pred = best['model'].predict(X_te)
            results_list.append({
                'Building': b_id, 'Model': name, 'Set': set_name,
                **calculate_metrics(y_test, y_pred, y_train_mean),
                'Train_s': best['fit_s'],
                'Infer_ms_per_sample': inference_ms_per_sample(lambda: best['model'].predict(X_te), len(X_te)),
                'Size_MB': model_size_mb((best['scaler'], best['model'])),
                'Best_params': str(best['params']),
            })
    print(f"Done: {b_id}")

results_p1 = pd.DataFrame(results_list)
results_p1.head()

Done: Fox_education_Shawanda
Done: Bear_education_Babara
Done: Hog_education_Robert
Done: Hog_office_Marilyn
Done: Hog_office_Marlena
Done: Fox_office_Israel
Done: Rat_assembly_Viola
Done: Rat_assembly_Teddy
Done: Fox_assembly_Terrell
Done: Rat_public_Sharron
Done: Rat_public_Kermit
Done: Bear_public_Orville


,Building,Model,Set,MAE,RMSE,R2,CV_RMSE_%,Train_s,Infer_ms_per_sample,Size_MB,Best_params
0,Fox_education_Shawanda,Linear Regression,Set_A,10.323006,11.403360,-0.988800,82.970506,0.003964,0.000051,0.001091,{}
1,Fox_education_Shawanda,Decision Tree,Set_A,11.830446,13.798208,-1.911863,100.395345,0.010956,0.000308,0.019004,"{'max_depth': 10, 'min_samples_leaf': 50}"
2,Fox_education_Shawanda,Random Forest,Set_A,11.839966,13.846338,-1.932212,100.745542,1.224830,0.015492,4.016815,"{'max_depth': 10, 'min_samples_leaf': 10}"
3,Fox_education_Shawanda,Linear Regression,Set_B,1.282700,2.230039,0.923941,16.225696,0.005132,0.000080,0.001293,{}
4,Fox_education_Shawanda,Decision Tree,Set_B,1.541772,2.690393,0.889298,19.575219,0.067158,0.000453,0.042521,"{'max_depth': 10, 'min_samples_leaf': 10}"


In [ ]:
# ==============================================================================
# SECTION 3.4: CROSS-BUILDING SUMMARY (Median & IQR) & EXPORT
# ==============================================================================
summary_p1 = results_p1.groupby(['Model', 'Set']).agg(
    MAE_Median=('MAE', 'median'),         MAE_IQR=('MAE', iqr),
    RMSE_Median=('RMSE', 'median'),       RMSE_IQR=('RMSE', iqr),
    R2_Median=('R2', 'median'),           R2_IQR=('R2', iqr),
    CV_RMSE_Median=('CV_RMSE_%', 'median'), CV_RMSE_IQR=('CV_RMSE_%', iqr),
    Train_s_Median=('Train_s', 'median'),
    Infer_ms_Median=('Infer_ms_per_sample', 'median'),
    Size_MB_Median=('Size_MB', 'median'),
).reset_index()

results_p1.to_csv(os.path.join(OUT_DIR, 'paper1_building_level_results.csv'), index=False)
summary_p1.to_csv(os.path.join(OUT_DIR, 'paper1_summary_results.csv'), index=False)

pd.set_option('display.width', 250, 'display.max_columns', None)
summary_p1.round(3)

,Model,Set,MAE_Median,MAE_IQR,RMSE_Median,RMSE_IQR,R2_Median,R2_IQR,CV_RMSE_Median,CV_RMSE_IQR,Train_s_Median,Infer_ms_Median,Size_MB_Median
0,Decision Tree,Set_A,13.317,7.951,18.062,7.604,0.291,1.010,26.619,20.023,0.011,0.000,0.018
1,Decision Tree,Set_B,4.042,3.980,7.002,6.379,0.907,0.202,10.340,6.859,0.069,0.000,0.043
2,Decision Tree,Set_C,4.262,3.905,7.086,6.620,0.903,0.206,9.690,7.427,0.075,0.000,0.042
3,Decision Tree,Set_W,13.199,8.524,17.642,8.464,0.401,1.167,25.764,23.697,0.021,0.000,0.019
4,Linear Regression,Set_A,17.966,11.403,21.234,12.534,0.013,1.252,37.673,21.957,0.004,0.000,0.001
5,Linear Regression,Set_B,3.246,6.099,5.059,9.856,0.946,0.034,6.168,8.013,0.004,0.000,0.001
6,Linear Regression,Set_C,3.211,6.133,5.003,9.773,0.946,0.036,6.162,7.757,0.007,0.000,0.002
7,Linear Regression,Set_W,16.859,7.398,19.261,8.185,0.280,0.929,29.065,24.112,0.005,0.000,0.001
8,Random Forest,Set_A,13.582,8.285,18.440,7.800,0.267,1.047,26.380,20.630,0.759,0.013,4.217
9,Random Forest,Set_B,3.842,4.059,6.203,7.591,0.937,0.125,8.780,8.552,3.792,0.013,6.159


### **3.5 Đo RAM khi huấn luyện và suy luận**
> Mỗi phép đo chạy trong một tiến trình Python mới để đo sạch; fit lại với đúng siêu tham số đã chọn ở 3.3 (không tinh chỉnh lại).

In [ ]:
# ==============================================================================
# SECTION 3.5: RAM USAGE (đỉnh RSS) KHI HUẤN LUYỆN VÀ SUY LUẬN (chỉ số chi phí Paper 1)
# Mỗi phép đo chạy trong một tiến trình Python MỚI để đo sạch (không bị nhiễu bởi bộ nhớ đã dùng trước đó).
# Fit lại với đúng siêu tham số đã chọn ở Section 3, nên không tinh chỉnh lại và không tính lại độ chính xác.
# Chạy khoảng 10-15 phút.
# ==============================================================================
import sys, json, subprocess, tempfile, ast

CHILD = """
import sys, os, gc, time, json, threading, ast
import numpy as np, psutil
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
P = psutil.Process(os.getpid())
rss = lambda: P.memory_info().rss / 1e6
d = np.load(sys.argv[1]); name = sys.argv[2]; params = ast.literal_eval(sys.argv[3])
Xtr, ytr, Xte = d['Xtr'], d['ytr'], d['Xte']
if name == 'Linear Regression':
    sc = StandardScaler().fit(Xtr); Xtr = sc.transform(Xtr); Xte = sc.transform(Xte)
makers = {'Linear Regression': lambda: LinearRegression(),
          'Decision Tree': lambda: DecisionTreeRegressor(random_state=42, **params),
          'Random Forest': lambda: RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, **params)}
model = makers[name]()
def peak(fn):
    gc.collect(); base = rss(); box = [base]; stop = threading.Event()
    def run():
        while not stop.is_set():
            box[0] = max(box[0], rss()); time.sleep(0.001)
    t = threading.Thread(target=run, daemon=True); t.start()
    fn(); stop.set(); t.join(); box[0] = max(box[0], rss())
    return box[0] - base
tr = peak(lambda: model.fit(Xtr, ytr))
inf = peak(lambda: model.predict(Xte))
print(json.dumps({'train': tr, 'infer': inf}))
"""
tmp_dir = tempfile.mkdtemp()
child_path = os.path.join(tmp_dir, 'child_ram.py')
with open(child_path, 'w') as f:
    f.write(CHILD)
npz_path = os.path.join(tmp_dir, 'data.npz')

res_ram = pd.read_csv(os.path.join(OUT_DIR, 'paper1_building_level_results.csv'))
ram_rows = []
for b_id in df_clean['building_id'].unique():
    b_train = train_df[train_df['building_id'] == b_id].sort_values('timestamp')
    b_test  = test_df[test_df['building_id'] == b_id]
    for set_name, cols in FEATURE_SETS.items():
        np.savez(npz_path, Xtr=b_train[cols].values, ytr=b_train['y_kwh'].values, Xte=b_test[cols].values)
        for name in ['Linear Regression', 'Decision Tree', 'Random Forest']:
            sel = (res_ram['Building'] == b_id) & (res_ram['Model'] == name) & (res_ram['Set'] == set_name)
            params = ast.literal_eval(res_ram.loc[sel, 'Best_params'].iloc[0])
            out = subprocess.run([sys.executable, child_path, npz_path, name, str(params)],
                                 capture_output=True, text=True, check=True)
            m = json.loads(out.stdout.strip().splitlines()[-1])
            ram_rows.append({'Building': b_id, 'Model': name, 'Set': set_name,
                             'RAM_train_peak_MB': m['train'], 'RAM_infer_peak_MB': m['infer']})
    print(f"RAM done: {b_id}")

ram_df = pd.DataFrame(ram_rows)
ram_summary = ram_df.groupby(['Model', 'Set']).agg(
    RAM_train_peak_MB_Median=('RAM_train_peak_MB', 'median'),
    RAM_infer_peak_MB_Median=('RAM_infer_peak_MB', 'median'),
).reset_index()
ram_df.to_csv(os.path.join(OUT_DIR, 'paper1_ram_building_level.csv'), index=False)
ram_summary.to_csv(os.path.join(OUT_DIR, 'paper1_ram_summary.csv'), index=False)
print("RAM tăng thêm khi chạy (MB), median trên 12 tòa nhà; mô hình Linear Regression gồm cả bộ đệm khởi tạo của thư viện số học:")
display(ram_summary.round(2))

RAM done: Fox_education_Shawanda
RAM done: Bear_education_Babara
RAM done: Hog_education_Robert
RAM done: Hog_office_Marilyn
RAM done: Hog_office_Marlena
RAM done: Fox_office_Israel
RAM done: Rat_assembly_Viola
RAM done: Rat_assembly_Teddy
RAM done: Fox_assembly_Terrell
RAM done: Rat_public_Sharron
RAM done: Rat_public_Kermit
RAM done: Bear_public_Orville
RAM tăng thêm khi chạy (MB), median trên 12 tòa nhà; mô hình Linear Regression gồm cả bộ đệm khởi tạo của thư viện số học:


,Model,Set,RAM_train_peak_MB_Median,RAM_infer_peak_MB_Median
0,Decision Tree,Set_A,0.48,0.00
1,Decision Tree,Set_B,0.48,0.00
2,Decision Tree,Set_C,0.61,0.00
3,Decision Tree,Set_W,0.53,0.00
4,Linear Regression,Set_A,2.13,0.03
5,Linear Regression,Set_B,2.69,0.03
6,Linear Regression,Set_C,1.86,0.05
7,Linear Regression,Set_W,2.62,0.00
8,Random Forest,Set_A,5.81,0.20
9,Random Forest,Set_B,7.70,0.20


### **3.6 FEATURE ABLATION (Random Forest, tham số mặc định) - Huy**
> **Note:** Quantifies the marginal error-reduction contribution of individual autoregressive lag features ($lag_1$, $lag_{24}$, $lag_{168}$, $roll\_mean_{24}$) and meteorological factors by incrementally expanding the input feature set.

In [ ]:
ablation_configs = {
    'Ablation_0_TimeOnly': SET_A_TIME,
    'Ablation_1_PlusLag1': SET_A_TIME + ['lag_1'],
    'Ablation_2_PlusLag24': SET_A_TIME + ['lag_1', 'lag_24'],
    'Ablation_3_PlusLag168': SET_A_TIME + ['lag_1', 'lag_24', 'lag_168'],
    'Ablation_4_FullLags': SET_B_TIME_LAG,
    'Ablation_5_FullLagsWeather': SET_C_FULL
}

ablation_results = []

print("Running Feature Ablation Study on Random Forest Model...")

for b_id in buildings:
    b_train = train_df[train_df['building_id'] == b_id]
    b_test  = test_df[test_df['building_id'] == b_id]

    y_train = b_train['y_kwh'].values
    y_test  = b_test['y_kwh'].values
    y_train_mean = np.mean(y_train)

    for step_name, f_cols in ablation_configs.items():
        rf_ablation = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
        rf_ablation.fit(b_train[f_cols], y_train)
        preds = rf_ablation.predict(b_test[f_cols])

        metrics = calculate_ashrae_metrics(y_test, preds, y_train_mean)
        ablation_results.append({
            'Building': b_id,
            'Ablation_Step': step_name,
            'Feature_Count': len(f_cols),
            **metrics
        })

ablation_df = pd.DataFrame(ablation_results)
print("Feature Ablation Study completed successfully.")

Running Feature Ablation Study on Random Forest Model...
Feature Ablation Study completed successfully.


### **3.7 Ablation mở rộng (Đức)**
> Bổ sung Linear Regression và các bộ đặc trưng thêm từng đặc trưng riêng lẻ (`A + lag_1`, `A + lag_24`, `A + lag_168`, `A + roll_mean_24`, `A + weather`), bên cạnh các bước cộng dồn của Huy. Random Forest dùng tham số mặc định, giống phần của Huy.

In [ ]:
# ==============================================================================
# SECTION 3.7: ABLATION STUDY (thêm lần lượt từng đặc trưng độ trễ và nhóm thời tiết)
# ==============================================================================
ABLATION_SETS = {
    'A (time only)':                  SET_A_TIME,
    'A + lag_1':                      SET_A_TIME + ['lag_1'],
    'A + lag_24':                     SET_A_TIME + ['lag_24'],
    'A + lag_168':                    SET_A_TIME + ['lag_168'],
    'A + roll_mean_24':               SET_A_TIME + ['roll_mean_24'],
    'A + weather':                    SET_A_TIME + WEATHER,
    'A + lag_1 + lag_24':             SET_A_TIME + ['lag_1', 'lag_24'],
    'A + lag_1 + lag_24 + lag_168':   SET_A_TIME + ['lag_1', 'lag_24', 'lag_168'],
    'B (A + all lags)':               SET_A_TIME + LAGS,
    'C (A + all lags + weather)':     SET_A_TIME + LAGS + WEATHER,
}
ABLATION_MODELS = ['Linear Regression', 'Random Forest']   # RF dùng tham số mặc định cố định, không tinh chỉnh

abl_list = []
for b_id in buildings:
    b_train = train_df[train_df['building_id'] == b_id]
    b_test  = test_df[test_df['building_id'] == b_id]
    y_test = b_test['y_kwh'].values
    y_train_mean = b_train['y_kwh'].mean()
    for set_name, cols in ABLATION_SETS.items():
        for name in ABLATION_MODELS:
            best = fit_select(name, cols, b_train, grid=[{}])
            y_pred = best['model'].predict(best['prep'](b_test[cols]))
            abl_list.append({'Building': b_id, 'Model': name, 'Set': set_name,
                             **calculate_metrics(y_test, y_pred, y_train_mean)})
    print(f"Ablation done: {b_id}")

ablation_ext_df = pd.DataFrame(abl_list)
abl_summary = ablation_ext_df.groupby(['Model', 'Set'], sort=False).agg(
    RMSE_Median=('RMSE', 'median'), R2_Median=('R2', 'median'),
    CV_RMSE_Median=('CV_RMSE_%', 'median'), CV_RMSE_IQR=('CV_RMSE_%', iqr),
).reset_index()

# Mức thay đổi CV(RMSE) so với Set A (điểm phần trăm, âm = tốt hơn)
base_cv = abl_summary[abl_summary['Set'] == 'A (time only)'].set_index('Model')['CV_RMSE_Median']
abl_summary['Delta_CV_vs_A'] = abl_summary.apply(lambda r: r['CV_RMSE_Median'] - base_cv[r['Model']], axis=1)

ablation_ext_df.to_csv(os.path.join(OUT_DIR, 'paper1_ablation_building_level.csv'), index=False)
abl_summary.to_csv(os.path.join(OUT_DIR, 'paper1_ablation_summary.csv'), index=False)

abl_pivot = abl_summary.pivot(index='Set', columns='Model', values='CV_RMSE_Median').loc[list(ABLATION_SETS.keys())]
print("Median CV(RMSE) % theo bộ đặc trưng (ablation):")
abl_pivot.round(2)

Ablation done: Fox_education_Shawanda
Ablation done: Bear_education_Babara
Ablation done: Hog_education_Robert
Ablation done: Hog_office_Marilyn
Ablation done: Hog_office_Marlena
Ablation done: Fox_office_Israel
Ablation done: Rat_assembly_Viola
Ablation done: Rat_assembly_Teddy
Ablation done: Fox_assembly_Terrell
Ablation done: Rat_public_Sharron
Ablation done: Rat_public_Kermit
Ablation done: Bear_public_Orville
Median CV(RMSE) % theo bộ đặc trưng (ablation):


Model,Linear Regression,Random Forest
Set,,
A (time only),37.67,26.47
A + lag_1,6.40,10.44
A + lag_24,18.10,21.14
A + lag_168,20.04,23.94
A + roll_mean_24,15.08,17.44
A + weather,29.06,25.26
A + lag_1 + lag_24,6.31,9.31
A + lag_1 + lag_24 + lag_168,6.17,9.62
B (A + all lags),6.17,9.46


In [ ]:
# ==============================================================================
# SECTION 3.8: ABLATION FIGURE
# ==============================================================================
sns.set_theme(style='whitegrid', font='DejaVu Sans')
order = list(ABLATION_SETS.keys())
fig, ax = plt.subplots(figsize=(10, 6))
h = 0.38
for i, m in enumerate(ABLATION_MODELS):
    vals = [abl_pivot.loc[s, m] for s in order]
    ax.barh(np.arange(len(order)) + (i - 0.5) * h, vals, height=h, label=m, alpha=0.9)
ax.set_yticks(np.arange(len(order)))
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel('Median CV(RMSE) % across 12 buildings (Lower is better)', fontweight='bold')
ax.set_title('Ablation: Contribution of Lag and Weather Features', fontweight='bold')
ax.legend()
sns.despine(fig=fig)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'paper1_ablation.png'), dpi=300)
plt.show()

# **SECTION 4: SYNTHETIC ANOMALY INJECTION (SAI) PROTOCOL**
> **Note:** Implements a controlled synthetic anomaly injection protocol generating 3% ground truth anomalies across 12 buildings: Type I (Daytime Power Spikes: $2.0\times - 4.0\times$) and Type II (Off-Hour Nocturnal Leaks: $2.0\times - 3.0\times$, 3-hour duration). Exports `synthetic_anomalies_12_clean.csv`.

In [ ]:
# ==============================================================================
# 5. SYNTHETIC ANOMALY INJECTION PROTOCOL
# ==============================================================================
def inject_synthetic_anomalies(df_input: pd.DataFrame, anomaly_ratio: float = 0.03, seed: int = 42) -> pd.DataFrame:
    """Injects Type I Spikes and Type II Leaks with explicit Ground Truth labels."""
    np.random.seed(seed)
    df_out = df_input.copy()
    df_out['is_anomaly'] = 0
    df_out['anomaly_type'] = 'Normal'

    n_samples = len(df_out)
    n_anomalies = int(n_samples * anomaly_ratio)
    n_spikes = n_anomalies // 2
    n_leaks = n_anomalies - n_spikes

    # 1. Type I: Daytime Power Spikes (08:00 - 18:00)
    daytime_idx = df_out[(df_out['hour'] >= 8) & (df_out['hour'] <= 18)].index
    spike_indices = np.random.choice(daytime_idx, size=n_spikes, replace=False)

    for idx in spike_indices:
        df_out.loc[idx, 'y_kwh'] *= np.random.uniform(2.0, 4.0)
        df_out.loc[idx, 'is_anomaly'] = 1
        df_out.loc[idx, 'anomaly_type'] = 'Power_Spike'

    # 2. Type II: Off-Hour Nocturnal Leaks (00:00 - 05:00, 3 contiguous hours)
    night_idx = df_out[(df_out['hour'] >= 0) & (df_out['hour'] <= 3) & (df_out['is_anomaly'] == 0)].index
    leak_starts = np.random.choice(night_idx, size=n_leaks // 3, replace=False)

    for start_idx in leak_starts:
        pos = df_out.index.get_loc(start_idx)
        mult = np.random.uniform(2.0, 3.0)
        for offset in range(3):
            if pos + offset < len(df_out):
                curr_idx = df_out.index[pos + offset]
                df_out.loc[curr_idx, 'y_kwh'] *= mult
                df_out.loc[curr_idx, 'is_anomaly'] = 1
                df_out.loc[curr_idx, 'anomaly_type'] = 'Night_Leak'

    return df_out

# Execute Injection and Export Synthetic Dataset
df_synthetic = inject_synthetic_anomalies(df_clean, anomaly_ratio=0.03, seed=SEED)

synthetic_csv_path = os.path.join(DATA_OUT_DIR, "synthetic_anomalies_12_clean.csv")
df_synthetic.to_csv(synthetic_csv_path, index=False)

print(f"Synthetic Anomaly Injection complete.")
print(f" - Output File: {synthetic_csv_path}")
print(f" - Total Anomalies Injected: {df_synthetic['is_anomaly'].sum():,} ({df_synthetic['is_anomaly'].mean()*100:.2f}%)")
print(f" - Breakdown: {df_synthetic['anomaly_type'].value_counts().to_dict()}")

# **SECTION 5: METRICS AGGREGATION & EXPORT**
> **Note:** Tổng hợp Trung vị (Median) và IQR trên 12 tòa nhà, kèm Naive baseline, số tòa nhà thắng/thua, kết quả theo loại hình tòa nhà và biểu đồ đánh đổi giữa độ chính xác và chi phí. Toàn bộ file kết quả lưu trong `03_Code_Colab/paper1_results`.

In [ ]:
# ==============================================================================
# SECTION 5.1: ABLATION RANDOM FOREST (HUY) - MEDIAN/IQR, EXPORT & ĐỐI CHIẾU VỚI BẢN MỞ RỘNG
# ==============================================================================
ablation_df.to_csv(os.path.join(OUT_DIR, 'paper1_ablation_rf_building_level.csv'), index=False)
abl_rf_summary = ablation_df.groupby('Ablation_Step', sort=False).agg(
    Feature_Count=('Feature_Count', 'first'),
    MAE_Median=('MAE', 'median'),               MAE_IQR=('MAE', iqr),
    RMSE_Median=('RMSE', 'median'),             RMSE_IQR=('RMSE', iqr),
    R2_Median=('R2', 'median'),                 R2_IQR=('R2', iqr),
    CV_RMSE_Median=('CV_RMSE_%', 'median'),     CV_RMSE_IQR=('CV_RMSE_%', iqr),
).reset_index()
abl_rf_summary.to_csv(os.path.join(OUT_DIR, 'paper1_ablation_rf_summary.csv'), index=False)
display(abl_rf_summary.round(3))

# Đối chiếu: Random Forest mặc định của Huy phải trùng với các dòng Random Forest cùng bộ đặc trưng ở bản mở rộng (3.7)
step_map = {'Ablation_0_TimeOnly': 'A (time only)', 'Ablation_1_PlusLag1': 'A + lag_1',
            'Ablation_2_PlusLag24': 'A + lag_1 + lag_24', 'Ablation_3_PlusLag168': 'A + lag_1 + lag_24 + lag_168',
            'Ablation_4_FullLags': 'B (A + all lags)', 'Ablation_5_FullLagsWeather': 'C (A + all lags + weather)'}
chk = ablation_df.assign(Set=ablation_df['Ablation_Step'].map(step_map)).merge(
    ablation_ext_df[ablation_ext_df['Model'] == 'Random Forest'], on=['Building', 'Set'], suffixes=('_huy', '_duc'))
print(f"Đối chiếu Ablation Random Forest (Huy so với Đức): {len(chk)} cặp, "
      f"chênh lệch CV(RMSE) lớn nhất = {np.abs(chk['CV_RMSE_%_huy'] - chk['CV_RMSE_%_duc']).max():.6f}")

In [ ]:
# ==============================================================================
# SECTION 5.2: ACCURACY vs COMPUTATIONAL COST TRADE-OFF (RQ2)
# ==============================================================================
colors = {'Linear Regression': '#1b9e77', 'Decision Tree': '#d95f02', 'Random Forest': '#7570b3'}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Accuracy vs Cost (each point = model x feature set; label = A/B/W/C)', fontweight='bold')

for _, r in summary_p1.iterrows():
    tag = r['Set'].split('_')[1]
    for ax, xcol in ((ax1, 'Infer_ms_Median'), (ax2, 'Size_MB_Median')):
        ax.scatter(r[xcol], r['CV_RMSE_Median'], color=colors[r['Model']], s=60)
        ax.annotate(tag, (r[xcol], r['CV_RMSE_Median']), textcoords='offset points', xytext=(5, 4), fontsize=9)

for m, c in colors.items():
    ax1.scatter([], [], color=c, label=m)
ax1.legend(fontsize=9)
ax1.set_xscale('log'); ax2.set_xscale('log')
ax1.set_xlabel('Median inference time (ms per sample, log)', fontweight='bold')
ax2.set_xlabel('Median model size (MB, log)', fontweight='bold')
for ax in (ax1, ax2):
    ax.set_ylabel('Median CV(RMSE) %', fontweight='bold')
sns.despine(fig=fig)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(OUT_DIR, 'paper1_tradeoff.png'), dpi=300)
plt.show()
print("Saved to:", OUT_DIR)

In [ ]:
# ==============================================================================
# SECTION 5.3: CONSOLIDATED TABLE (kèm Naive baseline), WIN COUNTS & BY BUILDING TYPE
# (Ô này đọc lại các file CSV đã lưu, nên chạy độc lập sau khi đã chạy ô nạp dữ liệu ở Section 2)
# ==============================================================================
import sys, sklearn
print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | seed {SEED}")

def iqr(x):
    return np.percentile(x, 75) - np.percentile(x, 25)

try:
    BASELINE_CSV = f"{ROOT}/03_Code_Colab/baseline_results/baseline_building_level_results.csv"
except NameError:  # chạy ngoài Colab
    BASELINE_CSV = os.environ.get("BASELINE_CSV", "baseline_building_level_results.csv")

cols = ['Building', 'Model', 'Set', 'MAE', 'RMSE', 'R2', 'CV_RMSE_%']
res  = pd.read_csv(os.path.join(OUT_DIR, 'paper1_building_level_results.csv'))
base = pd.read_csv(BASELINE_CSV)

# Hai mô hình Naive là mốc so sánh bắt buộc của Paper 1; chúng không dùng bộ đặc trưng nào -> Set = 'Reference'
naive = base[base['Model'].str.contains('Naive')].assign(Set='Reference')
all_res = pd.concat([naive[cols], res[cols]], ignore_index=True)
all_res['Type'] = all_res['Building'].str.split('_').str[1]   # loại hình lấy từ tên tòa nhà (education/public/assembly/office)

# --- 11.1 Bảng tổng hợp: median (IQR) trên 12 tòa nhà ---
def med_iqr(x, d=2):
    return f"{np.median(x):.{d}f} ({iqr(x):.{d}f})"

final_table = all_res.groupby(['Model', 'Set']).agg(
    MAE=('MAE', med_iqr), RMSE=('RMSE', med_iqr),
    R2=('R2', lambda x: med_iqr(x, 3)), CV_RMSE_pct=('CV_RMSE_%', med_iqr),
).reset_index()
order = ['Seasonal Naive (t-24)', 'Naive (t-1)', 'Linear Regression', 'Decision Tree', 'Random Forest']
final_table['o'] = final_table['Model'].map({m: i for i, m in enumerate(order)})
final_table = final_table.sort_values(['o', 'Set']).drop(columns='o').reset_index(drop=True)
final_table.to_csv(os.path.join(OUT_DIR, 'paper1_final_table.csv'), index=False)
print("Median (IQR) trên 12 tòa nhà, tập Test:")
display(final_table)

# --- 11.2 Đếm số tòa nhà mà mô hình X có CV(RMSE) thấp hơn mô hình Y ---
piv = all_res.pivot_table(index='Building', columns=['Model', 'Set'], values='CV_RMSE_%')
LR, DT, RF, NV = 'Linear Regression', 'Decision Tree', 'Random Forest', 'Naive (t-1)'
pairs = [((LR, 'Set_B'), (NV, 'Reference')), ((DT, 'Set_B'), (NV, 'Reference')), ((RF, 'Set_B'), (NV, 'Reference')),
         ((RF, 'Set_B'), (LR, 'Set_B')), ((RF, 'Set_B'), (DT, 'Set_B')), ((LR, 'Set_B'), (DT, 'Set_B')),
         ((LR, 'Set_C'), (LR, 'Set_B')), ((DT, 'Set_C'), (DT, 'Set_B')), ((RF, 'Set_C'), (RF, 'Set_B')),
         ((LR, 'Set_W'), (LR, 'Set_A')), ((DT, 'Set_W'), (DT, 'Set_A')), ((RF, 'Set_W'), (RF, 'Set_A'))]
n_b = piv.shape[0]
wins = pd.DataFrame([{'X (tốt hơn?)': f"{a[0]} {a[1]}", 'so với Y': f"{b[0]} {b[1]}",
                      'Số tòa X < Y': f"{int((piv[a] < piv[b]).sum())}/{n_b}"} for a, b in pairs])
wins.to_csv(os.path.join(OUT_DIR, 'paper1_win_counts.csv'), index=False)
display(wins)

# --- 11.3 Theo loại hình tòa nhà (mỗi loại chỉ có 3 tòa -> chỉ mô tả, không suy diễn thống kê) ---
by_type = all_res.groupby(['Type', 'Model', 'Set'])['CV_RMSE_%'].median().unstack(['Model', 'Set']).round(2)
by_type.to_csv(os.path.join(OUT_DIR, 'paper1_by_building_type.csv'))
print("Median CV(RMSE) % theo loại hình (3 tòa mỗi loại), Set B và Naive:")
display(by_type.loc[:, [c for c in by_type.columns if c[1] in ('Set_B', 'Reference')]])

## **Kết quả chính Paper 1 (12 tòa nhà, tập Test H2/2017)**
*Median trên 12 tòa nhà; chi tiết trong `paper1_final_table.csv`, `paper1_win_counts.csv`, `paper1_by_building_type.csv`, `paper1_ram_summary.csv`.*

| Mô hình (Set B) | CV(RMSE) % | R² | Huấn luyện (s) | Suy luận (ms/mẫu) | Kích thước (MB) |
|---|---|---|---|---|---|
| Naive (t-1) | 8.09 | 0.906 | – | – | – |
| Linear Regression | 6.17 | 0.946 | 0.004 | 0.00006 | 0.001 |
| Decision Tree | 10.34 | 0.907 | 0.07 | 0.0004 | 0.04 |
| Random Forest | 8.78 | 0.937 | 3.79 | 0.013 | 6.16 |

- **RQ1 (lag):** đặc trưng độ trễ cải thiện mạnh nhất. CV(RMSE) từ Set A xuống Set B: Linear Regression 37.67% → 6.17%, Decision Tree 26.62% → 10.34%, Random Forest 26.38% → 8.78%. Số tòa nhà có R² âm giảm từ 6/12 (Linear Regression) và 4/12 (Decision Tree, Random Forest) xuống 0/12.
- **RQ2 (nhóm đặc trưng):** `lag_1` đóng góp nhiều nhất, riêng nó đưa Linear Regression từ 37.67% xuống 6.40%. Thời tiết một mình chỉ giúp vừa phải (Set W so với Set A: Linear Regression 37.67% → 29.06%, tốt hơn ở 11/12 tòa; Decision Tree 26.62% → 25.76%; Random Forest 26.38% → 24.61%). Khi đã có lag, thời tiết gần như không thêm gì (Set C so với Set B: Linear Regression 6.17% → 6.16%, Random Forest 8.78% → 8.72%, Decision Tree 10.34% → 9.69%; Set C chỉ tốt hơn ở 6–7/12 tòa).
- **RQ3 (cân bằng sai số và chi phí):** Linear Regression tốt nhất về median CV(RMSE) và rẻ nhất; chỉ riêng Linear Regression vượt Naive (t-1) về median (6.17% so với 8.09%, thắng ở 10/12 tòa). Random Forest thắng Naive ở 8/12 tòa và Decision Tree ở 7/12 tòa, nhưng median của cả hai vẫn cao hơn Naive. Linear Regression thắng Decision Tree ở 9/12 tòa; Random Forest thắng Decision Tree ở 10/12 tòa và thắng Linear Regression ở 8/12 tòa. Random Forest nặng hơn Linear Regression khoảng 4.800 lần và suy luận chậm hơn khoảng 0 lần.
- **RAM:** RAM tăng thêm khi huấn luyện (median, Set B): Decision Tree 0.48 MB, Linear Regression 2.69 MB (gồm bộ đệm khởi tạo của thư viện số học), Random Forest 7.70 MB (tối đa khoảng 82 MB ở các tòa được chọn cây không giới hạn độ sâu); RAM khi suy luận không đáng kể (tối đa 0.2 MB). Mỗi phép đo chạy trong một tiến trình riêng; chi tiết `paper1_ram_summary.csv`.
- **Theo loại hình (mỗi loại 3 tòa, chỉ mang tính mô tả):** Random Forest tốt nhất ở nhóm education (4.60% so với Linear Regression 5.67%) nhưng kém ở nhóm office (12.57% so với 5.91%) và public (7.37% so với 4.93%). Hai tòa `Hog_office_Marilyn` (Random Forest 16.91%, Linear Regression 4.71%) và `Rat_public_Kermit` (19.18% so với 4.58%) là nơi cây thua nặng nhất. Mức tải trung bình của hai tòa này giảm mạnh từ Train sang Test (Rat_public_Kermit: 24.96 → 8.74; Hog_office_Marilyn: 41.56 → 28.31), trong khi cây quyết định chỉ dự báo trong khoảng giá trị đã thấy khi huấn luyện; Linear Regression và Naive bám theo giá trị giờ trước nên ít bị ảnh hưởng. Trên 12 tòa, mức chênh của Random Forest so với Linear Regression tương quan 0.86 với độ lệch mức tải giữa Train và Test (mô tả, chưa chứng minh nhân quả).
- **Ghi chú phương pháp:** siêu tham số chọn bằng TimeSeriesSplit (3 fold) trên tập Train, Test chỉ dùng để báo cáo, tập Val chưa dùng; Random Forest trong ablation dùng tham số mặc định (Set B: 9.46%), không so sánh trực tiếp với Random Forest đã tinh chỉnh ở trên; thời gian suy luận là trung bình theo lô trên cả tập Test; thời gian đo thay đổi nhẹ giữa các lần chạy.

